# 1002 23일차

## 1. 텍스트 데이터셋

단어를 번호로 바꿔 둔 케라스 문장 데이터. 정수 인코딩까지 끝난 상태로 받음

| | `reuters` | `imdb` |
|---|---|---|
| 내용 | 뉴스 기사 | 영화 리뷰 |
| 문제 | 주제 46개 다중분류 | 긍정·부정 이진분류 |
| train / test | 8982 / 2246 | 25000 / 25000 (이미 나뉨) |
| 출력층 | `Dense(46, softmax)` | `Dense(1, sigmoid)` |

1. `num_words` : 빈도 상위 단어만 남김. 나머지는 2로 바뀜
2. `maxlen` : 이 길이 이상인 문장을 뺌. 자르지 않음
3. `test_split` : test 비율. `reuters`에만 있음

- 0 패딩, 1 문장 시작, 2 사전에 없는 단어
- x는 길이가 다른 리스트를 담은 1차원 배열이라 `pad_sequences`로 2차원을 만듦
- `Embedding`의 `input_dim`은 `num_words`와 같게

### 1-1. 패딩 길이 정하기

평균보다 조금 길게 잡음. 최대 길이에 맞추면 대부분이 0

| | 최대 | 최소 | 평균 | `maxlen` |
|---|---|---|---|---|
| `reuters` | 2376 | 13 | 145.5 | 200 |
| `imdb` | | | 238.7 | 250 |

- 더 긴 문장은 `truncating='pre'`로 앞부분이 잘림

#### cf) maxlen이 두 군데

1. `load_data(maxlen=)` : 긴 문장을 뺌. 샘플 수가 줄어듦
2. `pad_sequences(maxlen=)` : 긴 문장은 자르고 짧은 문장은 채움. 샘플 수는 그대로

## 2. 텍스트 분류 모델

`Embedding` 출력은 3차원이라 2차원으로 줄인 뒤 `Dense`를 붙임

1. `Embedding` + `LSTM` / `GRU` : 마지막 칸 결과만 내보냄
2. `Embedding` + `Bidirectional` : 두 방향을 이어 붙임. `GRU(64)` → `(N, 128)`
3. `Embedding` + `Flatten` : 한 줄로 폄. 단어 순서는 보지 않음

- 순환층 `units`는 보통 64~128. `LSTM(1000)`은 `LSTM(64)`보다 파라미터가 약 100배

| | 목표 | test acc | loss |
|---|---|---|---|
| `reuters` | 0.67 | 0.739 | 3.50 |
| `imdb` | 0.9 | 0.835 > 0.849 | 5.22 > 5.67 |

- `imdb`는 `num_words`를 2000 → 10000으로 늘려 올라감
- loss가 큰 이유 : `EarlyStopping` 없이 1000 epoch를 다 돌아 과적합된 가중치로 평가함

### 2-1. 모델별 차원

| | DNN | CNN | RNN | Embedding |
|---|---|---|---|---|
| x | 2 | 4 | 3 | 2 |
| `input_shape` | 1 | 3 | 2 | 1 |
| output | 2 | 4 | 2 | 3 |

- `input_shape`는 x보다 하나 적음. 맨 앞 N을 뺌
- RNN은 `return_sequences=True`면 output 3
- Embedding만 차원이 늘어서 뒤에 RNN이나 `Flatten`이 옴

#### cf) 3·4차원 출력에 바로 붙인 Dense

`Dense`는 마지막 축에만 적용되어 차원이 그대로 남음

| 앞 층 출력 | 바로 붙인 `Dense` | 결과 |
|---|---|---|
| `Conv2D` `(N, 26, 26, 32)` | `Dense(10)` | `(N, 26, 26, 10)` |
| `Embedding` `(N, 250, 100)` | `Dense(1)` | `(N, 250, 1)` |

- 모델은 만들어지고 `fit`에서 정답과 모양이 안 맞아 에러
- 이미지는 `Flatten`·GAP, 텍스트는 순환층·`Flatten`을 먼저 넣음

## 3. 텐서플로와 넘파이

넘파이는 데이터 준비, 텐서플로는 학습

| | 넘파이 | 텐서플로 |
|---|---|---|
| 배열 | `ndarray` | `Tensor` |
| 계산 장치 | CPU | CPU, GPU |
| 미분 | 못 함 | 자동으로 함 |

- `pad_sequences`, `to_categorical`은 넘파이로 짜여 있어 결과가 `ndarray`
- `fit`에 넣은 넘파이는 텐서로 바뀌어 계산됨

## 4. sparse_categorical_crossentropy

y를 원핫하지 않고 정수 라벨 그대로 쓰는 다중분류 loss

| | `categorical_crossentropy` | `sparse_categorical_crossentropy` |
|---|---|---|
| y | 원핫 `(N, 10)` | 라벨 `(N,)`, `(N, 1)` |
| 정답을 라벨로 | `argmax` | 하지 않음 |

- 출력층과 계산은 같고 y 모양만 다름
- 정답에 `argmax`를 남기면 `AxisError`

## 5. DNN·CNN을 RNN으로

곱은 그대로 두고 `(N, timesteps, feature)`로 나눔

| 바꾸기 전 | RNN 모양 | 읽는 방식 |
|---|---|---|
| DNN `(20640, 8)` | `reshape(-1, 8, 1)` | 특성 8개를 8칸으로 |
| 흑백 `(N, 28, 28, 1)` | `reshape(-1, 28, 28)` | 28줄, 줄마다 28픽셀 |
| 컬러 `(N, 32, 32, 3)` | `reshape(-1, 32, 32 * 3)` | 32줄, 줄마다 32픽셀 × 3채널 |

1. 스케일링 → reshape 순서
2. train·val·test·예측 데이터를 모두 같은 모양으로
3. CNN에서 바꿀 때 `Flatten`·GAP를 뺌

- `reshape` 결과는 다시 담아야 바뀜
- DataFrame은 `.values` 후 reshape. 스케일러를 거치면 이미 넘파이
- `flow`는 4차원만 받음. 증폭한 뒤 3차원으로

| | CNN | RNN |
|---|---|---|
| mnist | 0.9894 | 0.9861 |
| fashion | 0.9029 | 0.8553 |
| cifar10 | 0.7756 | 0.3849 |

- RNN은 이미지를 한 줄씩 읽어 위아래 픽셀 관계를 순환으로만 이어 받음. 이미지가 복잡할수록 차이가 큼